In [0]:
from pyspark.sql import functions as F
storage_account = "dlsua5816bd"
login = "mariiasirska"
catalog = "dbr_dev_ua5816bd"

bronze = f"{catalog}.{login}_bronze"
raw = f"abfss://{login}@{storage_account}.dfs.core.windows.net/olist/raw"

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

def bronze_table(name):
    return f"{bronze}.olist_{name}_bronze"

display(dbutils.fs.ls(raw))

Batch 1: full load of the raw CSV files (idempotent)

In [0]:
def add_bronze_meta(df, source_file_col, batch_id):
    return (
        df.withColumn("source_file", source_file_col)
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("load_date", F.current_date())
        .withColumn("batch_id", F.lit(batch_id))
    )

def batch_loaded(tbl, batch_id):
    return (
        spark.catalog.tableExists(tbl)
        and spark.table(tbl).filter(F.col("batch_id") == batch_id).limit(1).count() > 0
    )

def load_batch1(name, file):
    tbl = bronze_table(name)
    if batch_loaded(tbl, 1):
        print(f"{tbl}: batch 1 already loaded, skipping")
        return

    df = (
        spark.read.option("header", True)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"{raw}/{file}")
    )
    cols = [F.col(f"`{c}`").alias(c.replace("\ufeff", "").strip()) for c in df.columns]
    df = df.select(*cols, F.col("_metadata.file_path").alias("source_file"))
    df = (
        df.withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("load_date", F.current_date())
        .withColumn("batch_id", F.lit(1))
    )
    df.write.mode("append").saveAsTable(tbl)
    print(f"{tbl}: loaded")

for name, file in files.items():
    load_batch1(name, file)

Batch 2: simulated source changes:

In [0]:
def b1(name):
    return (
        spark.table(bronze_table(name))
        .filter("batch_id = 1")
        .drop("source_file", "ingestion_ts", "load_date", "batch_id")
    )

def write_batch2(name, df):
    tbl = bronze_table(name)
    if batch_loaded(tbl, 2):
        print(f"{tbl}: batch 2 already loaded, skipping")
        return
    add_bronze_meta(df, F.lit("simulated_batch_2"), 2).write.mode("append").saveAsTable(tbl)
    print(f"{tbl}: batch 2 rows =", spark.table(tbl).filter("batch_id = 2").count())


In [0]:
orders = b1("orders")
changed = orders.orderBy(F.rand(42)).limit(3000).withColumn("order_status", F.lit("canceled"))
dups = changed.limit(500)
new_orders = (
    orders.orderBy(F.rand(7)).limit(200)
    .withColumn("order_id", F.concat(F.lit("new_"), F.col("order_id")))
)
write_batch2("orders", changed.unionByName(dups).unionByName(new_orders))


In [0]:
customers = b1("customers")
upd = (
    customers.orderBy(F.rand(1)).limit(200)
    .withColumn("customer_city", F.lit("rio de janeiro"))
    .withColumn("customer_state", F.lit("RJ"))
)
write_batch2("customers", upd)

sellers = b1("sellers").filter("seller_state <> 'SP'")
upd = (
    sellers.orderBy(F.rand(2)).limit(100)
    .withColumn("seller_city", F.lit("sao paulo"))
    .withColumn("seller_state", F.lit("SP"))
)
write_batch2("sellers", upd)

In [0]:
products = b1("products").filter("product_category_name IS NOT NULL")
upd = (
    products.orderBy(F.rand(3)).limit(100)
    .withColumn("product_category_name", F.lit("utilidades_domesticas"))
)
write_batch2("products", upd)

rows = []
for name in files:
    for r in spark.table(bronze_table(name)).groupBy("batch_id").count().collect():
        rows.append((name, int(r["batch_id"]), r["count"]))
display(spark.createDataFrame(rows, "table string, batch_id int, rows long").orderBy("table", "batch_id"))

In [0]:
def write_batch3(name, df):
    tbl = bronze_table(name)
    if batch_loaded(tbl, 3):
        print(f"{tbl}: batch 3 already loaded, skipping")
        return
    add_bronze_meta(df, F.lit("simulated_batch_3"), 3).write.mode("append").saveAsTable(tbl)

write_batch3("orders", b1("orders").limit(3).withColumn("order_status", F.lit("unknown_status")))
write_batch3("order_items", b1("order_items").limit(3).withColumn("price", F.lit("-5")))
write_batch3("order_reviews", b1("order_reviews").limit(3).withColumn("review_score", F.lit("9")))